### Programming for Biomedical Informatics
#### Week 9 - Functional Analysis Using Ontologies

Ontologies are commonly used to aid interpretation of molecular data, most commonly through use of functional annotations to genes and proteins using the Gene Onotlogy combined with downstream likelihood/enrichment analysis using tools such as GSEA as we have discussed. Ontologies are also used in strategies to align unstructured data with domains, for example looking for words and/or phrases that can be mapped to classes in ontologies. Examples here would include things like looking for terms associated with clinical terminolgies in patient discharge summaries.

In this notebook we will perform some basic phenotype extraction from publication abstracts, attempting to find examples of HPO terms that are associated with mentions of particular diseases.

To do this we will randomly select 1000 papers from PubMed that are tagged with the MeSH Major Topic "Autism Spectrum Disorder" retreive their titles and abstracts and then search for phenotypes and genes.

In [16]:
'''
we're going to use a biomedical named entity recognition model called en_core_sci_sm which is a model developed by the Allen Institute for biomedical text processing
https://allenai.github.io/scispacy/
'''

# %pip install scispacy
# %pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

"\nwe're going to use a biomedical named entity recognition model called en_core_sci_sm which is a model developed by the Allen Institute for biomedical text processing\nhttps://allenai.github.io/scispacy/\n"

In [17]:
# Setup for NLP

import pandas as pd
import spacy
from scispacy.linking import EntityLinker
from scispacy.abbreviation import AbbreviationDetector

# supress warnings
import warnings
warnings.filterwarnings('ignore')

## SciSpacy:
- The website shows what entities in biomedical data the transformer model has been trained on
- In Disease, it will have seen text that is used to describe them e.g. learned to assosiate entities and synonyms 
- Sci spaCy tells that a bit of text is a gene, not what gene it is
- You need to normalise to the gene name from that text
- The UMLS entity linker then goes on from that text to link to the UMLS to get the gene name

In [ ]:
'''when this is first run it will download some large data files to perform the UMLS linking
approx 2GB in total. On subsequent rune the model will take about 1 minute to load'''

# load the model
nlp = spacy.load("en_core_sci_sm"); # Not restricted to biomedical data, but in fine tuning it has seen biomedial data

# add abbreviations detector
nlp.add_pipe("abbreviation_detector"); # adding a modules here to deal with abbreviations, this adds ability to disambiguate abreviations from base model

# add UMLS entity-linker
nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True,"linker_name": "umls","filter_for_definitions": False});
# This module is a linker, once this is 

In [ ]:
# let's look at a simple example to understand this, this is an abstract from a paper. What concepts live in this abstract
queryText = 'PAX6 and GLI3 is a highly conserved transcription factor that plays a critical role in eye development in all animals. Mutations in the PAX6 gene are associated with aniridia, a congenital eye malformation characterized by the absence of the iris and other eye abnormalities.'

concepts = dict()

try:
    #perform nlp
    doc = nlp(queryText) # scispacy object, this returns a string of objects
    for entity in doc.ents:
        link = concept,score = entity._.kb_ents[0]
        concepts[entity.text] = concept
        print(entity.text, concept, score)
except:
    #case of no text
    pass


### This shows that Pax6 was linked to that number, these links are UMLS accession key
# You could chose only the child terms of genes in the UMLS ontology, and throw away the UMLS accessions not related to this

PAX6 C1418276 0.9495171904563904
GLI3 C0082708 0.9556154608726501
eye development C1517080 0.9721701741218567
animals C0003062 0.973741888999939
Mutations C0026882 0.9732257723808289
PAX6 gene C1418276 0.893574059009552
associated with C0332281 0.9891291856765747
aniridia C0003076 0.9635230302810669
congenital eye malformation C0015393 0.9502196311950684
characterized C1880022 0.9680080413818359
absence C0332197 0.9857189655303955
iris C0022077 0.9765904545783997
eye abnormalities C0015393 0.9852664470672607


In [20]:
# the human phenotype ontology contains database_cross_reference entries that include the UMLS concept id
# we can use this to link the HPO terms to the UMLS concepts

# load the HPO data using pronto
import pronto

# load the HPO ontology
# fetch the Human Phenotype Onology OBO file and parse it with pronto

# download the HPO ontology OBO file
import urllib.request

current_hpo_url = 'http://purl.obolibrary.org/obo/hp.obo'

# download the file
urllib.request.urlretrieve(current_hpo_url,'hpo.obo');

# parse the file
hpo = pronto.Ontology('hpo.obo')


In [ ]:
# we can look in the xrefs (sic. cross-references) of a term to find the UMLS concept id
# This is a link out from Gene Phenotype Ontology 
# FOr every concept found, I know its a human phenotype terms
# I can link between human phenotype term to UMLS terms
def hpo2concept(hpo_id):
    term = hpo[hpo_id]
    xrefs = [xref.id for xref in term.xrefs]
    try:
        umls_id = [xref for xref in xrefs if xref.startswith('UMLS')][0].split(':')[1]
        return umls_id
    except:
        return None

# let's test this function
hpo2concept('HP:0001695')

'C0018790'

In [22]:
# we're now going to brute force the conversion of all HPO terms to UMLS concepts
hpo2umls = {term.id:hpo2concept(term.id) for term in hpo.terms()}

In [23]:
# look at the first 10 entries
list(hpo2umls.items())[:10]

[('HP:0000001', 'C0444868'),
 ('HP:0000002', 'C4025901'),
 ('HP:0000394', 'C0266614'),
 ('HP:0000395', 'C1845272'),
 ('HP:0000396', 'C1837731'),
 ('HP:0000399', 'C4021806'),
 ('HP:0000400', 'C1835581'),
 ('HP:0000402', 'C0395837'),
 ('HP:0000403', 'C0747085'),
 ('HP:0000405', 'C0018777')]

In [36]:
#lets see if any of the CUIs from our test sentence have been mapped to HPO terms
for entity in concepts.keys():
    concept = concepts[entity]
    if concept in hpo2umls.values():
        print(entity, concept, [hpo[k] for k,v in hpo2umls.items() if v == concept])

aniridia C0003076 [Term('HP:0000526', name='Aniridia')]
congenital eye malformation C0015393 [Term('HP:0000478', name='Abnormality of the eye')]
eye abnormalities C0015393 [Term('HP:0000478', name='Abnormality of the eye')]


In [25]:
'''
As a niche example (not a mainstream ontology) we will look at the ASDPTO ontology which
has been developed as a custom ontology for autism spectrum disorder.

For every term in the ASDPTO ontology, we will look for UMLS concepts.

NB we are limited to the work done by ASDPTO curators in adding annotations to the terms
'''

current_asdpto_url = 'https://data.bioontology.org/ontologies/ASDPTO/submissions/1/download?apikey=4a2fbff0-ef88-432e-b1a1-dffc07e71146'

# download the file
urllib.request.urlretrieve(current_asdpto_url,'autism.obo');

# parse the file
autism  = pronto.Ontology('autism.obo')


In [26]:
# function to find the UMLS concept for a term in ASDPTO
def find_concept(term):
    for annotation in term.annotations:
        try:
            # if the string contains a cui= then it is a UMLS concept
            # extract the CUI
            if 'cui=' in annotation.resource:
                #split the string on 'cui=' and take the remainder
                concept = annotation.resource.split('cui=')[1]
                return(concept)
        except:
            pass

# we can now use this function to find the UMLS concept for each term in the ASDPTO ontology
asdpto2umls = {term.name:find_concept(term) for term in autism.terms()}

# remove any None entries
asdpto2umls = {k:v for k,v in asdpto2umls.items() if v is not None}

# print how many terms have been mapped to UMLS concepts
print(f'There are ',len(asdpto2umls),' terms in the ASDPTO ontology that have been mapped to UMLS concepts')

# look at the first 10 entries
list(asdpto2umls.items())[:10]

There are  82  terms in the ASDPTO ontology that have been mapped to UMLS concepts


[('Psychiatric Hospitalization', 'C0748061'),
 ('Ability to Convey Feelings', 'C1821397'),
 ('Gross Motor Skills', 'C0678858'),
 ('Perinatal Exposures', 'C1531967'),
 ('Use of Free Time', 'C0517871'),
 ('Depression', 'C0011570'),
 ('Medical History', 'C0262926'),
 ('Social Anxiety', 'C0424166'),
 ('Newborn Exposures', 'C0920206'),
 ('Adherence to Rules in the Home', 'C0562410')]

In [27]:
'''Now that we have all the NLP components in place to identify and map HPO terms let's now fetch the data and perform the analysis'''

"Now that we have all the NLP components in place to identify and map HPO terms let's now fetch the data and perform the analysis"

In [28]:
# let's use our knowledge of eUtils to fetch the raw material for our analysis
# we will use the requests library to fetch the data using the eUtils API
# we will use the xml library to parse the data

import urllib.request
import xml.etree.ElementTree as ET

# load my API key from the file
with open('../../bio_api_keys/ncbi.txt', 'r') as file:
    api_key = file.read().strip()

with open('../../bio_api_keys/ncbi_email.txt', 'r') as file:
    email = file.read().strip()

pubmed_query = '"Autism Spectrum Disorder[Majr]"'

# Define the parameters for the eSearch request
esearch_params = {
    'db': 'pubmed',
    'term': pubmed_query,
    'api_key': api_key,
    'email': email,
    'usehistory': 'y'
}

# encode the parameters so they can be passed to the API
encoded_data = urllib.parse.urlencode(esearch_params).encode('utf-8')

# the base request url for eSearch
url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

# make the request
request = urllib.request.Request(url, data=encoded_data)
response = urllib.request.urlopen(request)

# read into an XML object
esaerch_data_XML = ET.fromstring(response.read())

# print the number of results
count = esaerch_data_XML.find('Count').text
print(f'Total number of results: {count}')

# Extract WebEnv and QueryKey
webenv = esaerch_data_XML.find('WebEnv').text
query_key = esaerch_data_XML.find('QueryKey').text

efetch_params = {
'db': 'pubmed',
'query_key': query_key,
'WebEnv': webenv,
'retmax': '1000',
'api_key': api_key,
'email': email
}

# encode the parameters so they can be passed to the API
encoded_data = urllib.parse.urlencode(efetch_params).encode('utf-8')

# the base request url for eSearch
url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

# make the request
request = urllib.request.Request(url, data=encoded_data)
response = urllib.request.urlopen(request)

# read into an XML object
efetch_data_XML = ET.fromstring(response.read())

# let's look at the first 10 articles
for article in efetch_data_XML.findall('.//PubmedArticle')[:10]:
    pmid = article.find('.//PMID').text
    title = article.find('.//ArticleTitle').text
    abstract = article.find('.//AbstractText').text
    print(f'{pmid}: {title}')
    print(f'Abstract: {abstract}')

Total number of results: 45121
41226513: Urinary Uremic Toxin Signatures and the Metabolic Index of Gut Dysfunction (MIGD) in Autism Spectrum Disorder: A Stool-Phenotype-Stratified Analysis.
Abstract: Gut-derived uremic toxins may play a key role in neurodevelopmental conditions such as autism spectrum disorder (ASD) via host-microbe metabolic interactions. We evaluated five uremic toxins-p-cresyl sulfate (PCS), indoxyl sulfate (IS), trimethylamine N-oxide (TMAO), asymmetric dimethylarginine (ADMA), and symmetric dimethylarginine (SDMA)-in urine samples of 97 children with ASD and 71 neurotypical controls, stratified by Bristol Stool Chart (BSC) consistency types. Four of these toxins (PCS, IS, TMAO, ADMA) were integrated into a novel composite biomarker called the Metabolic Index of Gut Dysfunction (MIGD), while SDMA was measured as a complementary renal function marker. While individual metabolite levels showed no statistically significant differences, group-wise analysis by stool ph

In [ ]:
# for each article check whether it has an abstract and a title
# if it does combine the title and abstract into a single string
# if it doesn't remove it from the list
articles = dict()

for article in efetch_data_XML.findall('.//PubmedArticle'):
    try:
        pmid = article.find('.//PMID')
        title = article.find('.//ArticleTitle')
        abstract = article.find('.//AbstractText')
        tiab = title.text + ' ' + abstract.text
        articles[pmid.text] = tiab
    except:
        pass

print(f'Number of articles with abstracts: {len(articles)}') # 27 of these don't have abstracts

Number of articles with abstracts: 973


In [ ]:
# lets write a function based on code above to perform ner on articles and return
def nlp_article(article):

    current_article_concepts = dict()

    try:
        #perform nlp
        doc = nlp(article)
        for entity in doc.ents:
            link = concept,score = entity._.kb_ents[0]
            current_article_concepts[entity.text] = concept
    except:
        #case of no text
        pass

    ### We have now found the entities, and we need to check for a link

    hpo_terms = []

    for entity in current_article_concepts.keys():
        concept = current_article_concepts[entity]
        if concept in hpo2umls.values():
            print(entity, concept, [hpo[k] for k,v in hpo2umls.items() if v == concept])
            current_hpo = [hpo[k] for k,v in hpo2umls.items() if v == concept]
            hpo_terms.append(current_hpo[0].id)
    return list(set(hpo_terms))

In [ ]:
# now we can apply this function to all the articles
articles_hpo = {pmid: nlp_article(article) for pmid, article in articles.items()}

### We are not sure the importance of the references, just that they occur

autism C0004352 [Term('HP:0000717', name='Autism')]
autistic C0004352 [Term('HP:0000717', name='Autism')]
Autistic Mice C0004352 [Term('HP:0000717', name='Autism')]
autism-like behaviors C0856975 [Term('HP:0000729', name='Autistic behavior')]
autism C0004352 [Term('HP:0000717', name='Autism')]
autism C0004352 [Term('HP:0000717', name='Autism')]
attention deficit hyperactivity disorder C1263846 [Term('HP:0007018', name='Attention deficit hyperactivity disorder')]
ADHD C1263846 [Term('HP:0007018', name='Attention deficit hyperactivity disorder')]
attention deficit/hyperactivity disorder C1263846 [Term('HP:0007018', name='Attention deficit hyperactivity disorder')]
autism C0004352 [Term('HP:0000717', name='Autism')]
autistic adults C0004352 [Term('HP:0000717', name='Autism')]
Autism C0004352 [Term('HP:0000717', name='Autism')]
autistic brain C0004352 [Term('HP:0000717', name='Autism')]
Autism C0004352 [Term('HP:0000717', name='Autism')]
autism C0004352 [Term('HP:0000717', name='Autism')]


In [ ]:
# let's look at the results from the first 10 articles
list(articles_hpo.items())[:10]

# what percentage of articles have HPO terms
articles_with_hpo = [k for k,v in articles_hpo.items() if v]
print(f'Percentage of articles with HPO terms: {len(articles_with_hpo)/len(articles)*100:.2f}%')

## It may not be the case that phenotype terms may be in the paper at all e.g. purely molecular, or our system misses things

Percentage of articles with HPO terms: 37.20%


In [ ]:
# find the unique HPO terms found in the articles
unique_hpo_terms = list(set([term for terms in articles_hpo.values() for term in terms]))

# create a dataframe to store the data
df = pd.DataFrame(index=articles.keys(), columns=unique_hpo_terms)

# fill the dataframe
for pmid, terms in articles_hpo.items():
    df.loc[pmid, terms] = 1

# fill the NaN values with 0
df.fillna(0, inplace=True)

# print the first 5 rows
df.head()

## Their are cpu efficient ways to deal with the sparse matrices 

,HP:0030858,HP:0000952,HP:0001256,HP:0002063,HP:0000709,HP:0004395,HP:0001355,HP:0030650,HP:0000729,HP:0002015,...,HP:0002186,HP:0000938,HP:0002119,HP:0030646,HP:0012532,HP:0000618,HP:0007302,HP:0004905,HP:0012828,HP:0100753
41226513,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41224750,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41224335,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41221243,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41220090,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [34]:
# count the number of times each term appears and store this in a dataframe with columns
# 'HPO Term Name', 'HPO Term', 'Count' and sort by count
hpo_counts = df.sum().sort_values(ascending=False).reset_index()
hpo_counts.columns = ['HPO Term', 'Count']
hpo_counts['HPO Term Name'] = [hpo[term].name for term in hpo_counts['HPO Term']]

# use PrettyTable to display the data
from prettytable import PrettyTable

table = PrettyTable()
table.field_names = hpo_counts.columns
for row in hpo_counts.itertuples(index=False):
    table.add_row(row)
print(table)

+------------+-------+---------------------------------------------------+
|  HPO Term  | Count |                   HPO Term Name                   |
+------------+-------+---------------------------------------------------+
| HP:0000717 |  270  |                       Autism                      |
| HP:0007018 |   26  |      Attention deficit hyperactivity disorder     |
| HP:0001249 |   14  |              Intellectual disability              |
| HP:0001631 |   11  |                Atrial septal defect               |
| HP:0012837 |   9   |                    Generalized                    |
| HP:0000729 |   7   |                 Autistic behavior                 |
| HP:0030646 |   7   |                     Peripheral                    |
| HP:0012826 |   5   |                      Moderate                     |
| HP:0100753 |   5   |                   Schizophrenia                   |
| HP:0002958 |   4   |                Immune dysregulation               |
| HP:0009800 |   4   |   

## What have we done
- Loaded an ontology
- Queried pubmed, got back 45000
- Did entitiy recognition adn linking to UMLS
- Subsetted as phenotype
- 

In [ ]:
# optional additional
### Gene to phenotype database captures information about disordereers:
# Global catalogue of rare genetic diseases:
import pandas as pd

# lets speciifically look at all the DDG2P papers fom PubMed
# read in the DDG2P data file
ddg2p = pd.read_csv('./DDG2P.csv')
ddg2p.head()

# get the list of unique PMIDs from the 'pmids' column
pmids = ddg2p['pmids'].str.split(';').explode().unique()

# how many unique PMIDs are there
print(f'Number of unique PMIDs: {len(pmids)}')

# randomly select 1000 of these
import random
random.seed(42)
pmids_sample = random.sample(list(pmids), 1000)

import urllib.request
import xml.etree.ElementTree as ET

# load my API key from the file
with open('../../bio_api_keys/ncbi.txt', 'r') as file:
    api_key = file.read().strip()

with open('../../bio_api_keys/ncbi_email.txt', 'r') as file:
    email = file.read().strip()

# fetch the data for these PMIDs
# Define the parameters for the eSearch request
esearch_params = {
    'db': 'pubmed',
    'id': ','.join(pmids_sample),
    'api_key': api_key,
    'email': email,
    'usehistory': 'y'
}

# encode the parameters so they can be passed to the API
encoded_data = urllib.parse.urlencode(esearch_params).encode('utf-8')

# the base request url for eSearch
url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

# make the request
request = urllib.request.Request(url, data=encoded_data)
response = urllib.request.urlopen(request)

# read into an XML object
efetch_data_XML = ET.fromstring(response.read())

# for each article check whether it has an abstract and a title
# if it does combine the title and abstract into a single string
# if it doesn't remove it from the list
articles = dict()

for article in efetch_data_XML.findall('.//PubmedArticle'):
    try:
        pmid = article.find('.//PMID')
        title = article.find('.//ArticleTitle')
        abstract = article.find('.//AbstractText')
        tiab = title.text + ' ' + abstract.text
        articles[pmid.text] = tiab
    except:
        pass

print(f'Number of articles with abstracts: {len(articles)}')

# print the first 10 articles
for pmid, tiab in list(articles.items())[:10]:
    print(f'{pmid}: {tiab}')



Number of unique PMIDs: 6232
Number of articles with abstracts: 933
30809043: Evolutionary conserved networks of human height identify multiple Mendelian causes of short stature. Height is a heritable and highly heterogeneous trait. Short stature affects 3% of the population and in most cases is genetic in origin. After excluding known causes, 67% of affected individuals remain without diagnosis. To identify novel candidate genes for short stature, we performed exome sequencing in 254 unrelated families with short stature of unknown cause and identified variants in 63 candidate genes in 92 (36%) independent families. Based on systematic characterization of variants and functional analysis including expression in chondrocytes, we classified 13 genes as strong candidates. Whereas variants in at least two families were detected for all 13 candidates, two genes had variants in 6 (UBR4) and 8 (LAMA5) families, respectively. To facilitate their characterization, we established a clustered ne

In [40]:
# now we can apply this function to all the articles
articles_hpo = {pmid: nlp_article(article) for pmid, article in articles.items()}

short stature C0013336 [Term('HP:0003510', name='Severe short stature')]
Short stature C0013336 [Term('HP:0003510', name='Severe short stature')]
short stature genes C0013336 [Term('HP:0003510', name='Severe short stature')]
severe C0205082 [Term('HP:0012828', name='Severe')]
progressive C0205329 [Term('HP:0003676', name='Progressive')]
cortical atrophy C0235946 [Term('HP:0002120', name='Cerebral cortical atrophy'), Term('HP:0012444', name='Brain atrophy')]
dystopia canthorum C0423113 [Term('HP:0000506', name='Telecanthus')]
pigmentary changes C1260926 [Term('HP:0001000', name='Abnormality of skin pigmentation')]
hypotonia C0026827 [Term('HP:0001252', name='Hypotonia')]
intellectual disability C3714756 [Term('HP:0001249', name='Intellectual disability')]
craniofacial dysmorphism C0266617 [Term('HP:0000271', name='Abnormality of the face')]
short stature C0013336 [Term('HP:0003510', name='Severe short stature')]
skeletal anomalies C4021790 [Term('HP:0000924', name='Abnormality of the sk

In [41]:
# let's look at the results from the first 10 articles
list(articles_hpo.items())[:10]

# what percentage of articles have HPO terms
articles_with_hpo = [k for k,v in articles_hpo.items() if v]
print(f'Percentage of articles with HPO terms: {len(articles_with_hpo)/len(articles)*100:.2f}%')

Percentage of articles with HPO terms: 51.77%


In [42]:
# find the unique HPO terms found in the articles
unique_hpo_terms = list(set([term for terms in articles_hpo.values() for term in terms]))

# create a dataframe to store the data
df = pd.DataFrame(index=articles.keys(), columns=unique_hpo_terms)

# fill the dataframe
for pmid, terms in articles_hpo.items():
    df.loc[pmid, terms] = 1

# fill the NaN values with 0
df.fillna(0, inplace=True)

# print the first 5 rows
df.head()

,HP:0001847,HP:0003693,HP:0004923,HP:0001087,HP:0001288,HP:0000647,HP:0001955,HP:0004432,HP:0012826,HP:0000171,...,HP:0000508,HP:0000592,HP:0000486,HP:0003811,HP:0003477,HP:0001300,HP:0005505,HP:0001891,HP:0003241,HP:0000970
30809043,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
23542699,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
35607853,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
16752401,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
19015483,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [43]:
# count the number of times each term appears and store this in a dataframe with columns
# 'HPO Term Name', 'HPO Term', 'Count' and sort by count
hpo_counts = df.sum().sort_values(ascending=False).reset_index()
hpo_counts.columns = ['HPO Term', 'Count']
hpo_counts['HPO Term Name'] = [hpo[term].name for term in hpo_counts['HPO Term']]

# use PrettyTable to display the data
from prettytable import PrettyTable

table = PrettyTable()
table.field_names = hpo_counts.columns
for row in hpo_counts.itertuples(index=False):
    table.add_row(row)
print(table)

+------------+-------+--------------------------------------------------------+
|  HPO Term  | Count |                     HPO Term Name                      |
+------------+-------+--------------------------------------------------------+
| HP:0012828 |   84  |                         Severe                         |
| HP:0001249 |   54  |                Intellectual disability                 |
| HP:0003676 |   38  |                      Progressive                       |
| HP:0001250 |   29  |                        Seizure                         |
| HP:0003510 |   25  |                  Severe short stature                  |
| HP:0001252 |   23  |                       Hypotonia                        |
| HP:0000007 |   16  |            Autosomal recessive inheritance             |
| HP:0000006 |   15  |             Autosomal dominant inheritance             |
| HP:0000271 |   15  |                Abnormality of the face                 |
| HP:0200134 |   14  |                Ep

## Why:
- Are there certain genetic conditions that share genotypes
- Usually it is done by differential diagnosis e.g. similar set of diseases, but seperate on phenotype profile differences
- We are only using abstracts, but you can use this on 4,000,000 full text articles in open pubmed
- These will include patient level phenotype descriptions
